<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/0rating2inter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## preprocessing/

## Tiền xử lý bước 0

## Bước này tạo ra 3 file:

- i_id_mapping.csv: file map lại id của sản phẩm vì đã chuyển sang số
- u_id_mapping.csv: file map lại id của người dùng vì đã chuyển sang số
- sports14-indexed.inter: file tương tác tương tự file csv nhưng id của người và sản phẩm đã chuyển sang số


# 5-core filtering

- Extracting U-I interactions and performing 5-core, re-indexing
- dataset located at: http://jmcauley.ucsd.edu/data/amazon/links.html, rating only file in "Small" subsets for experimentation


In [3]:
import os
import pandas as pd

In [4]:
PATH = "./data/2014"

In [18]:
df = pd.read_parquet(os.path.join(PATH, "df_rating.parquet"))

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 160522 entries, 0 to 160521
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userID     160522 non-null  str    
 1   itemID     160522 non-null  str    
 2   rating     160522 non-null  float64
 3   timestamp  160522 non-null  int64  
dtypes: float64(1), int64(1), str(2)
memory usage: 8.5 MB


In [20]:
df.head(3)

,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000
2,A2LL1TGG90977E,097293751X,5.0,1395187200


## 5-core filtering


In [21]:
print(f"shape: {df.shape}")
df[:5]

shape: (160522, 4)


,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000
2,A2LL1TGG90977E,097293751X,5.0,1395187200
3,A5G19RYX8599E,097293751X,5.0,1376697600
4,A2496A4EWMLQ7,097293751X,4.0,1396310400


In [22]:
# Bỏ giá trị null với trùng
learner_id, course_id, tmstmp_str = "userID", "itemID", "timestamp"

df.dropna(subset=[learner_id, course_id, tmstmp_str], inplace=True)
df.drop_duplicates(subset=[learner_id, course_id, tmstmp_str], inplace=True)
print(f"After dropped: {df.shape}")
df[:3]

After dropped: (160522, 4)


,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000
2,A2LL1TGG90977E,097293751X,5.0,1395187200


In [25]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 5, 5


# Hàm này có nhiệm vụ tìm ra danh sách các ID "không hợp lệ" dựa trên số lượng tương tác.
def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {
        id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num
    }
    print(f"{len(ids)} illegal_ids_by_inter_num, field={field}")

    return ids


# Đây là hàm quan trọng nhất, thực hiện lọc dữ liệu theo vòng lặp cho đến khi mọi User và Item đều đạt chuẩn "5-core".
def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(
            df, field=learner_id, max_num=None, min_num=min_u_num
        )
        ban_items = get_illegal_ids_by_inter_num(
            df, field=course_id, max_num=None, min_num=min_i_num
        )
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if learner_id:
            dropped_inter |= df[learner_id].isin(ban_users)
        if course_id:
            dropped_inter |= df[course_id].isin(ban_items)
        print(f"{len(dropped_inter)} dropped interactions")
        df.drop(df.index[dropped_inter], inplace=True)

In [26]:
k_core = 5
filter_by_k_core(df)
print(f"k-core shape: {df.shape}")
print(f"shape after k-core: {df.shape}")
df[:2]

59 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160522 dropped interactions
0 illegal_ids_by_inter_num, field=userID
8 illegal_ids_by_inter_num, field=itemID
160286 dropped interactions
9 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160260 dropped interactions
0 illegal_ids_by_inter_num, field=userID
3 illegal_ids_by_inter_num, field=itemID
160225 dropped interactions
4 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160213 dropped interactions
0 illegal_ids_by_inter_num, field=userID
1 illegal_ids_by_inter_num, field=itemID
160197 dropped interactions
1 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
160193 dropped interactions
0 illegal_ids_by_inter_num, field=userID
0 illegal_ids_by_inter_num, field=itemID
k-core shape: (160189, 4)
shape after k-core: (160189, 4)


,userID,itemID,rating,timestamp
0,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
1,A19K65VY14D13R,097293751X,5.0,1372464000


## Re-index


In [27]:
df.reset_index(drop=True, inplace=True)

In [28]:
learner_id

'userID'

In [29]:
i_mapping_file = "i_id_mapping.csv"
u_mapping_file = "u_id_mapping.csv"

splitting = [0.8, 0.1, 0.1]
uid_field, iid_field = learner_id, course_id

# 1. Khai báo và lấy danh sách ID duy nhất
uni_users = pd.unique(df[uid_field])
uni_items = pd.unique(df[iid_field])

# 2. Tạo "Từ điển" ánh xạ (Mapping Dictionary)
# start from 0
u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

# 3. Cập nhật trực tiếp vào dữ liệu (Mapping)
df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

# 4. Lưu lại "Cuốn sổ địa chỉ" (Mapping File)
# dump
rslt_dir = PATH
u_df = pd.DataFrame(list(u_id_map.items()), columns=["user_id", "userID"])
i_df = pd.DataFrame(list(i_id_map.items()), columns=["asin", "itemID"])

u_df.to_csv(os.path.join(rslt_dir, u_mapping_file), sep="\t", index=False)
i_df.to_csv(os.path.join(rslt_dir, i_mapping_file), sep="\t", index=False)
print(f"mapping dumped...")

mapping dumped...


In [34]:
df.to_parquet(os.path.join(PATH, "df_inter.parquet"), index=False)

## Reload


In [35]:
indexed_df = pd.read_parquet(os.path.join(PATH, "df_inter.parquet"))
print(f"shape: {indexed_df.shape}")
indexed_df[:4]

shape: (160189, 4)


,userID,itemID,rating,timestamp
0,0,0,5.0,1373932800
1,1,0,5.0,1372464000
2,2,0,5.0,1395187200
3,3,0,5.0,1376697600


In [36]:
u_uni = indexed_df[learner_id].unique()
c_uni = indexed_df[course_id].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 19372
# of unique courses: 7025
min/max of unique learners: 0/19371
min/max of unique courses: 0/7024
